## **Generators**

A **generator** is a specialized type of function in Python that allows you to loop over a sequence of data **lazily** (one item at a time) instead of computing and storing the entire dataset in memory all at once.

Standard functions use the `return` keyword to output a final result and instantly terminate, completely destroying their local state. Generators use the **`yield`** keyword instead, which temporarily pauses the function's execution, hands a value back to the caller, and retains its exact state to resume precisely where it left off on the next request.

---

- **🏎️ The Core Problem: Memory Exhaustion**
    - Imagine you need to process a dataset containing 10 million integers. A traditional approach would calculate the entire list and load it into your system's RAM:

```python
# ❌ Dangerous: Creates 10 million integers in memory simultaneously
def get_massive_range(n):
    result = []
    for i in range(n):
        result.append(i)
    return result  # Returns a massive list

data = get_massive_range(10_000_000)  # Consumes hundreds of megabytes of RAM
```

> If you run this with a significantly higher limit, your system will run out of memory and crash (`MemoryError`).

---

- **💡 The Solution: Lazy Evaluation with `yield`**
    - A generator converts this memory-heavy pipeline into an iterator stream. It only computes a number when the loop explicitly demands it, dropping it from memory immediately afterward:

```python
# ⚡ Memory Efficient: Streams numbers one by one on demand
def get_massive_range_generator(n):
    for i in range(n):
        yield i  # Pauses execution and hands 'i' back safely

# Consumes practically ZERO memory initial setup!
data_stream = get_massive_range_generator(10_000_000)

for number in data_stream:
    if number > 3:
        break
    print(number)
# Output: 0, 1, 2, 3
```

---

- **🛠️ The Mechanics Under the Hood: `next()`**
    - When you call a generator function, it **does not execute any code inside the function body.** Instead, it creates and returns a **generator object**. To get values out of it, Python utilizes the `next()` function behind the scenes:

```python
def simple_steps():
    print("🎬 Step 1 starting")
    yield "A"
    print("🔄 Step 2 starting")
    yield "B"

# 1. Instantiate the generator object
gen = simple_steps()

# 2. Trigger execution up to the first yield
print(next(gen))  
# Output:
# 🎬 Step 1 starting
# A

# 3. Resume from step 1 up to the second yield
print(next(gen))  
# Output:
# 🔄 Step 2 starting
# B

# 4. Triggering next() again throws a StopIteration exception
# next(gen) -> ❌ Raises StopIteration (This tells loops when to stop)
```

---

- **🏗️ Method 2: Generator Expressions**
    - Just like Python features list comprehensions, it also offers **generator expressions**. They share an almost identical syntax, but utilize **parentheses `()**` instead of square brackets `[]`.

```python
# ❌ List Comprehension: Creates the entire list in memory right now
heavy_list = [x ** 2 for x in range(1000000)]

# ⚡ Generator Expression: Ready to calculate squares on the fly, using zero RAM
lazy_stream = (x ** 2 for x in range(1000000))

print(next(lazy_stream))  # Output: 0
print(next(lazy_stream))  # Output: 1
```

---

- **🚀 Advanced Feature: Two-Way Communication (`send()`)**
    - Generators aren't just for pulling data out; you can also push data back *into* them while they are paused using the **`.send()`** method. When you use `.send(value)`, the `yield` statement inside the generator receives that value as its assignment payload:

```python
def interactive_counter():
    count = 0
    while True:
        # Pauses here, outputs count. If a value is sent in, it overrides 'jump'
        jump = yield count
        if jump is not None:
            count += jump
        else:
            count += 1

counter = interactive_counter()

print(next(counter))      # Output: 0 (Initializes the generator)
print(next(counter))      # Output: 1
print(counter.send(10))   # Output: 11 (Jumps forward by 10!)
print(next(counter))      # Output: 12
```

---

- **⚖️ Summary Comparison: Lists vs. Generators**

| Feature | List / Collection | Generator |
| --- | --- | --- |
| **Evaluation Strategy** | Eager (Everything processed upfront) | Lazy (Processed on request) |
| **Memory Footprint** | Large (Scales linearly with data size) | Extremely Small (Constant space complexity) |
| **Access Patterns** | Can read indexes arbitrarily (`data[5]`) | Sequential access only via `next()` loops |
| **Reusability** | Infinite (Can be read over and over) | Exhaustible (Can only be read through **once**) |

### **Creating generators**

- **🟢 Advantages of Generators :**
    * **Syntactic Simplicity:** They are cleaner and simpler to write than traditional list-generating functions, eliminating the boilerplate code of creating an empty list, calling `list.append()`, and returning the final collection in favor of a single `yield` statement.
    * **Minimal Memory Footprint:** Items are evaluated and streamed one at a time, removing the need to store massive datasets concurrently in system RAM.
    * **Dynamic Dependencies:** Values are computed dynamically at the exact moment they are requested, allowing results to change based on real-time external variables (such as checking a live queue or stack).
    * **Lazy Evaluation:** Execution is entirely lazy; if your application only requests the first few results, the remaining elements are never calculated. Between these sequential requests, the generator's state is completely frozen.

---

- **🔴 Disadvantages of Generators :**
    * **Single-Use Exhaustion:** Results can only be iterated through exactly once. After a generator yields its final item, it is completely depleted and cannot be reset or reused.
    * **Unknown/Infinite Size:** The absolute size of a generator is hidden until processing concludes. Because a generator can theoretically be infinite, attempting to forcefully cast it into a standard list (`list(infinite_generator)`) can instantly exhaust system memory and crash the Python interpreter.
    * **No Sequence Slicing:** Because items are generated on-demand rather than indexed sequentially in memory, slicing syntax like `generator[10:20]` is completely unsupported (though it can be bypassed using `itertools.islice`, which discards skipped elements).
    * **No Direct Indexing:** Arbitrary random access is impossible; you cannot instantly pull a specific element from a generator using bracket notation like `generator[5]`.

In [1]:
def generator():
    yield 1
    yield 'a'
    yield []
    return 'result'

In [2]:
result = generator()

result

<generator object generator at 0x7fd1ff783d70>

In [3]:
list(result)

[1, 'a', []]

In [4]:
list(result)

[]

In [6]:
def generator_with_return():
    yield "some value"
    return "exit generator function"

result = generator_with_return()

In [8]:
next(result)

'some value'

In [9]:
def lazy():
    print("Before yielding")
    yield "yielding"
    print("After yielding")

In [10]:
generator = lazy()

next(generator)

Before yielding


'yielding'

In [13]:
generator = lazy()

next(generator)

Before yielding


'yielding'

In [14]:
try:
    next(generator)
except StopIteration:
    pass

After yielding


In [16]:
for item in lazy():
    print(item)

Before yielding
yielding
After yielding


### **Creating infinite generators**

In [18]:
def count(start=0, step=1, stop=None):
    n = start
    while stop is not None and n < stop:
        yield n
        n += step

In [19]:
list(count(10, 2.5, 20))

[10, 12.5, 15.0, 17.5]

> Due to the potentially infinite nature of generators, caution is required. Without the stop variable, simply doing **list(count())** would result in an infinite loop that results in an out-of-memory situation quite fast.

### **Generators wrapping iterables**

In [20]:
def square(iterable):
    for i in iterable:
        yield i**2

In [21]:
list(square(range(5)))

[0, 1, 4, 9, 16]

In [22]:
def square(iterable):
    yield 'Begin'
    for i in iterable:
        yield i**2
    yield 'End'

In [23]:
list(square(range(5)))

['Begin', 0, 1, 4, 9, 16, 'End']

In [24]:
def odd(iterable):
    for i in iterable:
        if i % 2:
            yield i

def square(iterable):
    for i in iterable:
        yield i**2 

In [25]:
list(square(odd(range(10))))

[1, 9, 25, 49, 81]

### **Generator comprehensions**

In [26]:
squares = (x**2 for x in range(5))

squares

<generator object <genexpr> at 0x7fd1fd482740>

In [27]:
list(squares)

[0, 1, 4, 9, 16]

In [28]:
import itertools

In [33]:
result = itertools.count()

odd = (x for x in result if x % 2)
sliced_odd = itertools.islice(odd, 5)

list(sliced_odd)

[1, 3, 5, 7, 9]

In [34]:
result = itertools.count()
sliced_result = itertools.islice(result, 5)
odd_result = (x for x in sliced_result if x % 2)

list(odd_result)

[1, 3]

### **Class-based generators and iterators**

A **coroutine** is an advanced evolution of a generator designed for high-performance **asynchronous programming and cooperative multitasking**.

While regular generators are pulling mechanisms (producing data via `yield`), coroutines are primarily data consumers or data processors. They allow execution to be paused and resumed dynamically, enabling Python to switch tasks and handle massive amounts of concurrent operations (like network requests or file inputs) without the heavy overhead of system threading.

---

- **🏎️ The Architectural Evolution: Generators vs. Coroutines**
    - The shift from a generator to a coroutine comes down to whether the data flows *out* or *in*:
        * **Generator (Producer):** Uses `yield value` to push data out to a loop.
        * **Coroutine (Consumer):** Uses `variable = yield` to pause execution and wait for data to be pushed *into* it from the outside.

---

- **🛠️ Classic Coroutines (The Native `yield` Method)**
    - Before Python introduced dedicated async keywords, coroutines were built natively using generators and the `.send()` method.

```python
def string_matcher(pattern):
    print(f"🎬 Coroutine initialized. Looking for: '{pattern}'")
    try:
        while True:
            # The coroutine pauses here, waiting for data to be sent in
            text = yield
            if pattern in text:
                print(f"🎯 Match Found: {text}")
    except GeneratorExit:
        print("🧹 Coroutine closed cleanly.")

# 1. Instantiate the coroutine
matcher = string_matcher("error")

# 2. Priming: Advance execution to the first yield statement
next(matcher)  # Output: 🎬 Coroutine initialized...

# 3. Stream data into the coroutine dynamically
matcher.send("System status: OK")      # (Ignored, doesn't contain 'error')
matcher.send("Warning: low memory")    # (Ignored)
matcher.send("Critical error detected") # Output: 🎯 Match Found: Critical error detected

# 4. Explicitly shut down the coroutine
matcher.close()  # Output: 🧹 Coroutine closed cleanly.
```

---

- **🚀 Modern Coroutines (`async` and `await`)**
    - In modern Python, the manual `yield`-based coroutine pattern is abstracted away into dedicated language syntax: **`async def`** and **`await`**. This framework powers high-concurrency engines like `asyncio`.
    - Instead of running linearly, an `async` coroutine yields control back to an underlying system event loop whenever it hits an `await` blocker (like a slow network download), allowing other code routines to execute in the meantime.

```python
import asyncio

async def fetch_api_data(endpoint, delay):
    print(f"📡 Requesting data from {endpoint}...")
    # Voluntarily releases control back to the event loop during this idle wait time
    await asyncio.sleep(delay) 
    print(f"✅ Received payload from {endpoint}")
    return {"status": 200, "endpoint": endpoint}

async def main():
    # Run multiple coroutines concurrently on a single system thread
    task1 = fetch_api_data("users_api", 2)
    task2 = fetch_api_data("orders_api", 1)
    
    # Gathers and runs them simultaneously
    results = await asyncio.gather(task1, task2)
    print("✨ All requests processed successfully.")

# Execute the event loop
asyncio.run(main())

# Output Order:
# 📡 Requesting data from users_api...
# 📡 Requesting data from orders_api...
# ✅ Received payload from orders_api   (Finishes first because delay was 1s)
# ✅ Received payload from users_api    (Finishes second)
# ✨ All requests processed successfully.
```

---

- **⚖️ Summary Comparison Matrix**

| Feature | Subroutines (Normal Functions) | Generators | Coroutines (`async/await`) |
| --- | --- | --- | --- |
| **Entry Points** | Single (Starts from the top line) | Single (Starts from the top line) | **Multiple** (Resumes exactly where it was paused) |
| **Data Interaction** | Receives inputs once at call time | Emits data out via `yield` | **Consumes and pauses** waiting for external data or events |
| **State Retention** | No (Destroyed upon reaching `return`) | Yes (Local variables are frozen during `yield`) | **Yes** (Maintains local variables and execution frame) |
| **Primary Use Case** | Basic sequential business logic | Memory-efficient data streaming | **High-concurrency IO** (Web servers, APIs, scrapers) |